# 03 — Modeling: baseline + 9 modeli + stacking + voting

**Modele:** Baseline, Logistic Regression (L1/L2/**ElasticNet**), Balanced RF, Random Forest, XGBoost, SVM, MLP (sklearn), **CatBoost**, **Stacking** (BRF+XGB+CB → LR), **Voting** (LR+SVM+NN).

**Pipeline:** feature engineering v4 (88 cech) → imputacja + OHE → SMOTE (poza CatBoost/Stacking/XGB/RF) → [scaler] → model.

**Regularyzacja LR:** GridSearch po `penalty ∈ {l1, l2, elasticnet}` z `l1_ratio ∈ {0.1, 0.3, 0.5, 0.7, 0.9}` (solver=saga). XGBoost: reg_alpha (L1) + reg_lambda (L2). MLP: alpha (L2). CatBoost: l2_leaf_reg.

**Optymalizacja:** GridSearchCV (LR, SVM) / RandomizedSearchCV 50 iteracji (drzewa, NN), 5-fold CV (ROC-AUC).

**Podział danych:** train_fit (48%) / val (12%) / test (40%) — próg dobierany na val, nigdy na danych testowych.

> Pełny trening może trwać **30–90 min**. Jeśli masz już `models/*.joblib`, ustaw `LOAD_ONLY = True`.

> Importuje wszystkie biblioteki i moduły projektu: `config`, `data_loader`, `features`, `evaluate`, `best_model`, `model`, `threshold`.

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
(ROOT / 'reports').mkdir(parents=True, exist_ok=True)

from config import F1_TARGET, MODELS_DIR, RANDOM_STATE, TEST_SIZE, VAL_SIZE
from data_loader import load_hr
from features import get_X_y
from evaluate import (
    compute_feature_importance,
    evaluate_models,
    plot_feature_importance,
    plot_mlp_learning_curve,
    plot_model_comparison,
    save_reports,
    summarize_cv_results,
)
from best_model import format_best_model_report, save_best_model
from model import train_all_models
from threshold import ThresholdClassifier

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

MODEL_LABELS = {
    'baseline':            'Baseline (najczęstsza klasa)',
    'logistic_regression': 'Logistic Regression',
    'balanced_rf':         'Balanced RF',
    'random_forest':       'Random Forest',
    'xgboost':             'XGBoost',
    'catboost':            'CatBoost',
    'svm':                 'SVM',
    'neural_network':      'MLP (sklearn)',
    'stacking':            'Stacking (BRF+XGB+CB+SVM / LR)',
}

## 1. Trenowanie lub wczytanie modeli

Ustaw `LOAD_ONLY = True`, aby pominąć trening i wczytać zapisane pipeline'y z `models/` (po `python scripts/train.py`).

> Ładuje dane i dzieli na train_fit/val/test. Jeśli `LOAD_ONLY=True` — wczytuje gotowe `.joblib` z `models/`; w przeciwnym razie trenuje wszystkie modele. Wyświetla wyniki CV i tabelę progów decyzyjnych.

In [2]:
LOAD_ONLY = True  # True = tylko wczytaj models/*.joblib

df = load_hr()
X, y, num_cols, cat_cols = get_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train,
)
print(f'Podział: train_fit={len(X_train_fit)} ({len(X_train_fit)/len(X):.0%}), '
      f'val={len(X_val)} ({len(X_val)/len(X):.0%}), '
      f'test={len(X_test)} ({len(X_test)/len(X):.0%})')

_SKIP_STEMS = {'metadata', 'best_model', 'preprocessing_pipeline', 'ensemble'}

if LOAD_ONLY and (MODELS_DIR / 'xgboost.joblib').exists():
    print('Wczytywanie modeli z models/...')
    pipelines = {}
    for path in sorted(MODELS_DIR.glob('*.joblib')):
        if path.stem in _SKIP_STEMS:
            continue
        pipelines[path.stem] = joblib.load(path)
    meta = joblib.load(MODELS_DIR / 'metadata.joblib')
    searches = {}
    threshold_sources = meta.get('threshold_sources', {})
    tuning_f1 = meta.get('tuning_f1_at_threshold', meta.get('oof_f1_at_threshold', {}))
    preprocessor_fitted = pipelines.get('xgboost', list(pipelines.values())[0]).pipeline.named_steps['preprocess']
    artifacts = {
        'pipelines': pipelines,
        'searches': searches,
        'X_train': X_train_fit,
        'X_test': X_test,
        'X_val': X_val,
        'y_train': y_train_fit,
        'y_test': y_test,
        'y_val': y_val,
        'num_cols': meta['num_cols'],
        'cat_cols': meta['cat_cols'],
        'preprocessor_fitted': preprocessor_fitted,
        'threshold_sources': threshold_sources,
        'oof_f1_scores': tuning_f1,
    }
else:
    print(f'Trenowanie wszystkich modeli (cel F1@prog >= {F1_TARGET})...')
    artifacts = train_all_models(verbose=True)
    pipelines = artifacts['pipelines']
    threshold_sources = artifacts.get('threshold_sources', {})
    tuning_f1 = artifacts['oof_f1_scores']
    X_test, y_test = artifacts['X_test'], artifacts['y_test']
    X_val, y_val = artifacts['X_val'], artifacts['y_val']

if artifacts.get('searches'):
    cv_summary = summarize_cv_results(artifacts['searches'])
    print('\nWyniki CV (5-fold, F1 na train_fit):')
    display(cv_summary.round(4))
else:
    cv_summary = None
    cv_path = ROOT / 'reports' / 'cv_summary.csv'
    if cv_path.exists():
        cv_summary = pd.read_csv(cv_path, index_col=0)
        print('\nWyniki CV (z reports/cv_summary.csv):')
        display(cv_summary.round(4))

thr_rows = []
for name, f1_tuning in tuning_f1.items():
    if name not in pipelines:
        continue
    pipe = pipelines[name]
    thr = getattr(pipe, 'threshold', 0.5)
    src = threshold_sources.get(name, getattr(pipe, 'threshold_source', '?'))
    f1_val_actual = None
    if src in ('oof', 'val') and name != 'baseline':
        try:
            from sklearn.metrics import f1_score as _f1
            y_pred_val = pipe.predict(X_val)
            f1_val_actual = round(_f1(y_val, y_pred_val, zero_division=0), 4)
        except Exception:
            pass
    thr_rows.append({
        '_key': name,
        'model': MODEL_LABELS.get(name, name),
        'F1@tuning': round(f1_tuning, 4),
        'prog': round(thr, 3),
        'zrodlo': src,
        'F1@val': f1_val_actual if f1_val_actual is not None else '—',
        f'cel>={F1_TARGET}': 'TAK' if f1_tuning >= F1_TARGET else 'nie',
    })

metrics_raw = evaluate_models(pipelines, X_test, y_test)
best_info = save_best_model(
    pipelines,
    metrics_raw,
    threshold_sources=threshold_sources,
    oof_f1_scores=tuning_f1,
    verbose=True,
)
print('\n' + format_best_model_report(best_info))

thr_df = pd.DataFrame(thr_rows)
thr_df['F1_test'] = thr_df['_key'].map(metrics_raw['f1']).round(4)
def _diff(row):
    if isinstance(row['F1@val'], float):
        return round(row['F1@val'] - row['F1_test'], 4)
    return '—'
thr_df['diff(val-test)'] = thr_df.apply(_diff, axis=1)
thr_df = thr_df.drop(columns=['_key'])
print('\nProgi: zrodlo doboru, F1@tuning, F1@val (uczciwy), F1@test (holdout):')
display(thr_df)

metrics = metrics_raw.rename(index=MODEL_LABELS)
print('\nMetryki TEST (holdout 40%, uczciwy zbior):')
display(metrics.round(4))
print(f"Najlepszy F1 na test: {metrics['f1'].idxmax()} = {metrics['f1'].max():.4f}")

Podział: train_fit=705 (48%), val=177 (12%), test=588 (40%)
Wczytywanie modeli z models/...

Wyniki CV (z reports/cv_summary.csv):


,best_cv_f1,best_params
model,,
baseline,0.0000,{}
logistic_regression,0.4991,"{'model__C': 0.0001, 'model__penalty': 'l2', '..."
balanced_rf,0.5595,"{'model__n_estimators': 300, 'model__min_sampl..."
random_forest,0.5595,"{'model__n_estimators': 300, 'model__min_sampl..."
neural_network,0.5664,"{'model__learning_rate_init': 0.002, 'model__h..."
svm,0.5493,"{'model__C': 0.01, 'model__gamma': 'scale', 'm..."
xgboost,0.6085,"{'model__subsample': 0.6, 'model__scale_pos_we..."
catboost,0.5852,"{'model__learning_rate': 0.02, 'model__l2_leaf..."
stacking,NaN,{}



NAJLEPSZY MODEL (best_model)
Model:              Logistic Regression (logistic_regression)
Kryterium wyboru:   najwyzszy F1 na zbiorze testowym (holdout 20%)
Plik pipeline:      models/best_model.joblib
Kopia z:            models/logistic_regression.joblib

Metryki na TEST:
  F1:         0.5775
  Precision:  0.5870
  Recall:     0.5684
  ROC-AUC:    0.8145
  Accuracy:   0.8656
  Prog dec.:  0.5572
  Zrodlo progu: oof
  F1@prog (tuning): 0.6146

Streamlit uzywa models/best_model.joblib jako model glowny.

Zapisano: C:\Users\joann\Documents\DataScience\LearnIT\PROJECT\models\best_model.joblib
Raport:   C:\Users\joann\Documents\DataScience\LearnIT\PROJECT\reports\best_model.txt

NAJLEPSZY MODEL (best_model)
Model:              Logistic Regression (logistic_regression)
Kryterium wyboru:   najwyzszy F1 na zbiorze testowym (holdout 20%)
Plik pipeline:      models/best_model.joblib
Kopia z:            models/logistic_regression.joblib

Metryki na TEST:
  F1:         0.5775
  Precision:  0.58

,model,F1@tuning,prog,zrodlo,F1@val,cel>=0.8,F1_test,diff(val-test)
0,Baseline (najczęstsza klasa),0.0000,0.500,most_frequent,—,nie,0.0000,—
1,Logistic Regression,0.6146,0.557,oof,0.5,nie,0.5775,-0.0775
2,Balanced RF,0.5854,0.473,oof,0.4783,nie,0.5000,-0.0217
3,Random Forest,0.5854,0.473,oof,0.4783,nie,0.5000,-0.0217
4,MLP (sklearn),0.5774,0.378,oof,0.5614,nie,0.5106,0.0508
5,SVM,0.5911,0.823,oof,0.5116,nie,0.5250,-0.0134
6,XGBoost,0.6154,0.509,oof,0.5079,nie,0.5024,0.0055
7,CatBoost,0.6055,0.536,oof,0.5091,nie,0.5185,-0.0094
8,Stacking (BRF+XGB+CB+SVM / LR),0.5700,0.542,oof,0.4898,nie,0.4842,0.0056



Metryki TEST (holdout 40%, uczciwy zbior):


,accuracy,precision,recall,f1,roc_auc
model,,,,,
Balanced RF,0.8469,0.5294,0.4737,0.5000,0.8041
Baseline (najczęstsza klasa),0.8384,0.0000,0.0000,0.0000,0.5000
CatBoost,0.8452,0.5213,0.5158,0.5185,0.8047
Logistic Regression,0.8656,0.5870,0.5684,0.5775,0.8145
MLP (sklearn),0.8435,0.5161,0.5053,0.5106,0.7754
Random Forest,0.8469,0.5294,0.4737,0.5000,0.8041
Stacking (BRF+XGB+CB+SVM / LR),0.8333,0.4842,0.4842,0.4842,0.8046
SVM,0.8707,0.6462,0.4421,0.5250,0.8140
XGBoost,0.8248,0.4643,0.5474,0.5024,0.8179


Najlepszy F1 na test: Logistic Regression = 0.5775


## 2. Najlepszy model (`best_model`)

Najwyzszy **F1 na holdout** → `models/best_model.joblib`, opis: `reports/best_model.txt`. Streamlit uzywa tego modelu.

## 3. Porownanie modeli

> Oblicza pełne metryki (F1, Precision, Recall, ROC-AUC, Accuracy) na zbiorze treningowym i testowym; wyświetla różnicę ΔF1 jako kontrolę overfittingu.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, accuracy_score

def _full_metrics(pipelines, X, y, label):
    rows = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)
        y_proba = pipe.predict_proba(X)[:, 1]
        rows.append({
            'model': MODEL_LABELS.get(name, name),
            'F1': round(f1_score(y, y_pred, zero_division=0), 4),
            'Precision': round(precision_score(y, y_pred, zero_division=0), 4),
            'Recall': round(recall_score(y, y_pred, zero_division=0), 4),
            'ROC-AUC': round(roc_auc_score(y, y_proba), 4),
            'Accuracy': round(accuracy_score(y, y_pred), 4),
        })
    df = pd.DataFrame(rows).set_index('model').sort_values('F1', ascending=False)
    print(f'\n{"="*60}')
    print(f'Metryki — {label}')
    print(f'{"="*60}')
    display(df)
    return df

metrics_train = _full_metrics(pipelines, X_train_fit, y_train_fit, 'ZBIÓR TRENINGOWY (train_fit)')
metrics_test  = _full_metrics(pipelines, X_test, y_test, 'ZBIÓR TESTOWY (holdout 40%)')

print('\n' + '='*60)
print('OVERFITTING — różnica F1 (train - test):')
print('  >0.10 = możliwy overfitting')
print('='*60)
diff = (metrics_train['F1'] - metrics_test['F1']).sort_values(ascending=False).round(4)
display(diff.rename('ΔF1 (train−test)').to_frame())

> Rysuje wykres słupkowy porównujący F1, Precision, Recall i ROC-AUC wszystkich modeli na zbiorze testowym i zapisuje PNG do `reports/`.

In [ ]:
plot_model_comparison(metrics)

fig, ax = plt.subplots(figsize=(12, 5))
metrics[['f1', 'precision', 'recall', 'roc_auc']].plot(kind='bar', ax=ax, rot=45)
ax.set_title('Metryki na zbiorze testowym (holdout 20%)')
ax.set_ylim(0, 1)
ax.axhline(F1_TARGET, color='crimson', ls='--', lw=1, label=f'cel F1={F1_TARGET}')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'model_comparison_notebook.png', dpi=120)
plt.show()
print('Najlepszy F1 (test):', metrics['f1'].idxmax(), '=', round(metrics['f1'].max(), 4))

## 3a. Raport szczegółowy — metryki per klasa attrition

Poniżej pełny raport dla każdego modelu z podziałem na klasy:
- **Zostaje (0)** — pracownicy, którzy pozostali w firmie (N=493 w teście)
- **Rezygnuje (1)** — pracownicy, którzy odeszli (N=95 w teście)

Metryki: **Precision**, **Recall**, **F1** dla każdej klasy + **ROC-AUC** (globalna miara separacji klas).

> Generuje per-klasowy raport klasyfikacji (F1, Precision, Recall, ROC-AUC) dla klas Zostaje/Rezygnuje i Macro avg — tworzy kolorowe tabele, wykresy słupkowe i zapisuje wyniki do `reports/detailed_per_class_report.csv`.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# ── Zbierz metryki per klasa dla każdego modelu ──────────────────────────────
rows = []
for name, pipe in pipelines.items():
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    cr  = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    auc = round(roc_auc_score(y_test, y_proba), 4)
    label = MODEL_LABELS.get(name, name)

    for cls_key, cls_label in [
        ('0', 'Zostaje (0)'),
        ('1', 'Rezygnuje (1)'),
        ('macro avg', 'Macro avg'),
    ]:
        rows.append({
            'Model':     label,
            'Klasa':     cls_label,
            'Precision': round(cr[cls_key]['precision'], 4),
            'Recall':    round(cr[cls_key]['recall'],    4),
            'F1':        round(cr[cls_key]['f1-score'],  4),
            'Support':   int(cr[cls_key]['support']),
            'ROC-AUC':   auc if cls_key == '1' else '—',
        })

detail_df = pd.DataFrame(rows)

# ── Tabela zbiorcza: widok pivot (modele × metryki) dla klasy Rezygnuje ──────
pivot_resign = (
    detail_df[detail_df['Klasa'] == 'Rezygnuje (1)']
    .set_index('Model')[['Precision', 'Recall', 'F1', 'ROC-AUC']]
    .sort_values('F1', ascending=False)
)
pivot_stay = (
    detail_df[detail_df['Klasa'] == 'Zostaje (0)']
    .set_index('Model')[['Precision', 'Recall', 'F1']]
    .sort_values('F1', ascending=False)
)
pivot_macro = (
    detail_df[detail_df['Klasa'] == 'Macro avg']
    .set_index('Model')[['Precision', 'Recall', 'F1']]
    .sort_values('F1', ascending=False)
)

# ── Wyświetl ──────────────────────────────────────────────────────────────────
print("=" * 65)
print("KLASA 1 — REZYGNUJE  (cel: wykryć odejście)")
print("  Support = 95 próbek testowych")
print("=" * 65)
display(
    pivot_resign.style
    .background_gradient(subset=['F1'],        cmap='RdYlGn', vmin=0, vmax=0.7)
    .background_gradient(subset=['Precision'], cmap='Blues',  vmin=0, vmax=0.75)
    .background_gradient(subset=['Recall'],    cmap='Purples',vmin=0, vmax=0.75)
    .background_gradient(subset=['ROC-AUC'],   cmap='Oranges',vmin=0.5, vmax=0.85)
    .format(precision=4)
)

print("\n" + "=" * 65)
print("KLASA 0 — ZOSTAJE  (dla kontekstu)")
print("  Support = 493 próbki testowe")
print("=" * 65)
display(
    pivot_stay.style
    .background_gradient(subset=['F1'],        cmap='RdYlGn', vmin=0.8, vmax=1.0)
    .background_gradient(subset=['Precision'], cmap='Blues',  vmin=0.8, vmax=1.0)
    .background_gradient(subset=['Recall'],    cmap='Purples',vmin=0.8, vmax=1.0)
    .format(precision=4)
)

print("\n" + "=" * 65)
print("MACRO AVG  (niważona średnia klas 0 i 1)")
print("=" * 65)
display(
    pivot_macro.style
    .background_gradient(subset=['F1'],        cmap='RdYlGn', vmin=0.4, vmax=0.85)
    .background_gradient(subset=['Precision'], cmap='Blues',  vmin=0.4, vmax=0.85)
    .background_gradient(subset=['Recall'],    cmap='Purples',vmin=0.4, vmax=0.85)
    .format(precision=4)
)

# ── Wykres: F1 per klasa side-by-side ────────────────────────────────────────
f1_plot = (
    detail_df[detail_df['Klasa'].isin(['Zostaje (0)', 'Rezygnuje (1)'])]
    .pivot(index='Model', columns='Klasa', values='F1')
    .sort_values('Rezygnuje (1)', ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(f1_plot))
width = 0.38
bars0 = ax.bar(x - width/2, f1_plot['Zostaje (0)'],   width, label='Zostaje (0)',    color='#2ecc71', alpha=0.85)
bars1 = ax.bar(x + width/2, f1_plot['Rezygnuje (1)'], width, label='Rezygnuje (1)', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(f1_plot.index, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('F1-score')
ax.set_title('F1 per klasa attrition — wszystkie modele (zbiór testowy)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.axhline(F1_TARGET, color='crimson', ls='--', lw=1.2, label=f'cel F1={F1_TARGET}')
ax.legend(loc='upper right')

for bar in [*bars0, *bars1]:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'f1_per_class_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Wykres: ROC-AUC per model ─────────────────────────────────────────────────
auc_series = (
    detail_df[detail_df['Klasa'] == 'Rezygnuje (1)']
    .set_index('Model')['ROC-AUC']
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#e74c3c' if v == auc_series.max() else '#3498db' for v in auc_series.values]
auc_series.plot(kind='bar', ax=ax, color=colors, alpha=0.85)
ax.set_title('ROC-AUC — wszystkie modele (zbiór testowy)', fontsize=12)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.4, 0.9)
ax.axhline(0.5, color='gray', ls='--', lw=1, label='losowy klasyfikator (0.5)')
ax.set_xticklabels(auc_series.index, rotation=30, ha='right', fontsize=9)
ax.legend()
for patch, val in zip(ax.patches, auc_series.values):
    ax.text(patch.get_x() + patch.get_width()/2, val + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'roc_auc_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Zapis raportu per klasa do CSV ────────────────────────────────────────────
detail_df.to_csv(ROOT / 'reports' / 'detailed_per_class_report.csv', index=False)
print("\nZapisano: reports/detailed_per_class_report.csv")
print("Zapisano: reports/f1_per_class_comparison.png")
print("Zapisano: reports/roc_auc_comparison.png")

## 3. Macierze pomylek

> Rysuje macierze pomyłek (TN, FP, FN, TP) dla wszystkich modeli na zbiorze testowym w układzie siatki 3×n i zapisuje PNG do `reports/`.

In [ ]:
items = list(pipelines.items())
n = len(items)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, (name, pipe) in zip(axes, items):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pipe.predict(X_test), ax=ax, cmap='Blues', colorbar=False,
    )
    ax.set_title(MODEL_LABELS.get(name, name), fontsize=10)

for ax in axes[len(items):]:
    ax.axis('off')

fig.suptitle('Macierze pomyłek — zbiór testowy', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Feature importance

Metody obliczania ważności cech — różne dla każdej klasy modeli:

| Model | Metoda | Interpretacja |
|-------|--------|---------------|
| **Logistic Regression** (L1/L2/ElasticNet) | `|coef_[0]|` | Bezwzględna wartość współczynnika; L1 zeruje nieistotne cechy |
| **Random Forest / Balanced RF** | `feature_importances_` (Gini) | Średni spadek zanieczyszczenia Gini po splicie |
| **XGBoost** | `feature_importances_` (gain) | Średni zysk informacji ze splitów na tej cesze |
| **CatBoost** | natywna ważność | PredictionValuesChange — ważność dla predykcji probabilistycznych |
| **MLP / SVM** | permutation importance | Spadek F1 po losowym przetasowaniu cechy (10 powtórzeń) |

> Uwaga: metody nie są bezpośrednio porównywalne między modelami — każda ma inną skalę i interpretację.

> Oblicza ważność cech dla każdego modelu metodą odpowiednią dla jego klasy (`|coef|`, Gini, gain, permutation), zapisuje CSV i PNG do `reports/`.

In [ ]:
# Oblicz feature importance — metoda zależy od klasy modelu
importance = compute_feature_importance(
    pipelines,
    artifacts['preprocessor_fitted'],
    artifacts['num_cols'],
    artifacts['cat_cols'],
    X_test,
    y_test,
)

# Zapisz CSV + wykresy PNG do reports/
reverse_labels = {v: k for k, v in MODEL_LABELS.items()}
metrics_save = metrics.copy()
metrics_save.index = [reverse_labels.get(i, i) for i in metrics_save.index]
save_reports(metrics_save, importance, cv_summary)

print(f"Importance obliczone dla: {list(importance.keys())}")
print("Pliki zapisane: reports/feature_importance_*.csv, reports/feature_importance_*.png")

> Rysuje poziome wykresy słupkowe top-20 najważniejszych cech dla każdego modelu osobno z wartościami liczbowymi i zapisuje PNG do `reports/`.

In [ ]:
## 4.1 Wykresy — Top 20 cech per model

METHOD_LABELS = {
    'logistic_regression': 'Regresja logistyczna — |coef| (L1/ElasticNet)',
    'balanced_rf':         'Balanced RF — Gini importance',
    'random_forest':       'Random Forest — Gini importance',
    'xgboost':             'XGBoost — gain importance',
    'catboost':            'CatBoost — PredictionValuesChange',
    'neural_network':      'MLP — permutation importance (F1)',
    'svm':                 'SVM — permutation importance (F1)',
}

TOP_N = 20

for model_key, series in importance.items():
    label = METHOD_LABELS.get(model_key, model_key)
    top = series.head(TOP_N).sort_values()
    n_nonzero = (series > 0).sum()

    fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.35)))
    colors = ['#c0392b' if v == top.max() else '#2980b9' for v in top.values]
    top.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'{label}\n(top {TOP_N} z {len(series)} cech, niezerowych: {n_nonzero})', fontsize=12)
    ax.set_xlabel('Ważność cechy')
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    for patch, val in zip(ax.patches, top.values):
        ax.text(
            max(val, 0) + top.max() * 0.01, patch.get_y() + patch.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8, color='#333'
        )
    plt.tight_layout()
    plt.savefig(ROOT / 'reports' / f'feature_importance_{model_key}_top{TOP_N}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"\nTop 5 — {label}:")
    display(series.head(5).rename('importance').to_frame())

## 4.2 Porównanie ważności cech między modelami

Które cechy są kluczowe we **wszystkich** modelach? Normalizujemy importance do [0,1] per model i tworzymy heatmapę top-15 cech (wg średniej rangi).

> Normalizuje ważność cech do [0,1] per model i rysuje heatmapę top-15 cech wg średniej rangi — pokazuje które cechy są ważne we wszystkich modelach jednocześnie.

In [ ]:
## 4.2 Heatmapa — unormowana ważność cech (top 15 wg średniej rangi)

SHORT_LABELS = {
    'logistic_regression': 'LR',
    'balanced_rf':         'BRF',
    'random_forest':       'RF',
    'xgboost':             'XGB',
    'catboost':            'CB',
    'neural_network':      'MLP',
    'svm':                 'SVM',
}

# Normalizacja [0, 1] per model
normed = {}
for k, s in importance.items():
    mn, mx = s.min(), s.max()
    normed[k] = (s - mn) / (mx - mn + 1e-12)

# Wszystkie unikalne cechy
all_feats = sorted(set().union(*[set(s.index) for s in normed.values()]))

# DataFrame: wiersze = cechy, kolumny = modele
df_imp = pd.DataFrame(
    {SHORT_LABELS.get(k, k): normed[k].reindex(all_feats, fill_value=0) for k in normed},
    index=all_feats,
)

# Wybierz top 15 cech wg średniej unormowanej ważności
top15_feats = df_imp.mean(axis=1).sort_values(ascending=False).head(15).index
df_top15 = df_imp.loc[top15_feats].sort_values(df_imp.columns[0], ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    df_top15,
    annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': 'Unormowana ważność [0–1]'},
    ax=ax,
)
ax.set_title('Porównanie ważności cech między modelami\n(top 15 wg średniej, znormalizowane do [0,1] per model)', fontsize=12)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'feature_importance_comparison_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nTop 15 cech (średnia unormowana ważność):")
display(df_top15.mean(axis=1).sort_values(ascending=False).rename('avg_normalized_importance').round(3).to_frame())

# Zapis CSV
df_top15.to_csv(ROOT / 'reports' / 'feature_importance_comparison.csv')
print("Zapisano: reports/feature_importance_comparison.csv")

> Tworzy tabelę rankingową top-10 cech per model (kolumny = modele, wiersze = pozycje w rankingu) i zapisuje do `reports/feature_importance_rankings.csv`.

In [ ]:
## 4.3 Tabela rankinków — top 10 cech per model

print("=" * 70)
print("TOP 10 CECH — RANKING PER MODEL")
print("=" * 70)

rows_rank = {}
for model_key, series in importance.items():
    label = SHORT_LABELS.get(model_key, model_key)
    rows_rank[label] = series.head(10).index.tolist()

max_len = max(len(v) for v in rows_rank.values())
rank_df = pd.DataFrame(
    {col: vals + [''] * (max_len - len(vals)) for col, vals in rows_rank.items()},
    index=[f'#{i+1}' for i in range(max_len)],
)
display(rank_df)

rank_df.to_csv(ROOT / 'reports' / 'feature_importance_rankings.csv')
print("\nZapisano: reports/feature_importance_rankings.csv")

## 5. Krzywa uczenia się — MLP (sklearn)

Krzywa uczenia pokazuje jak zmienia się **F1 na zbiorze treningowym i walidacyjnym CV** w zależności od liczby próbek treningowych.

- **Mały gap** trening–walidacja → model dobrze generalizuje (brak overfitting)
- **Duży gap** → model może się przeuczyć
- **Obie krzywe niskie** → underfitting (model ma zbyt małą pojemność lub za mało cech)

> Generuje krzywą uczenia (learning curve) dla MLP — pokazuje jak F1 zmienia się w zależności od liczby próbek treningowych przy 5-fold CV.

In [ ]:
# Krzywa uczenia się MLP — bazowy pipeline bez wrappera ThresholdClassifier
mlp_pipeline = pipelines['neural_network'].pipeline  # ImbPipeline: preprocess → smote → scaler → model

print('Generowanie krzywej uczenia (5-fold CV × 8 rozmiarów)...')
lc_path = plot_mlp_learning_curve(
    mlp_pipeline,
    X_train_fit,
    y_train_fit,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
print(f'Krzywa uczenia zapisana: {lc_path}')

from IPython.display import Image
Image(str(lc_path))

## 6. Wnioski

### Wyniki na zbiorze testowym (holdout 40%, 588 próbek) — finalne

Modele: **9** (baseline + LR + BRF + RF + NN + SVM + XGB + CatBoost + Stacking).

Feature engineering: **36 cech inżynierowanych** (19 v1/v2 + 17 v3), łącznie ~66 kolumn wejściowych.

| # | Model | F1 | Precision | Recall | ROC-AUC | Accuracy |
|---|-------|-----|-----------|--------|---------|----------|
| 1 | **Logistic Regression** | **0.5775** | 0.5870 | 0.5684 | 0.8145 | 0.8656 |
| 2 | SVM | 0.5250 | **0.6462** | 0.4421 | 0.8140 | **0.8707** |
| 3 | Stacking (BRF+XGB+CB→LR) | 0.5210 | 0.4336 | **0.6526** | 0.8046 | 0.8061 |
| 4 | CatBoost | 0.5165 | 0.5402 | 0.4947 | 0.8129 | 0.8503 |
| 5 | MLP (sklearn) | 0.5106 | 0.5161 | 0.5053 | 0.7754 | 0.8435 |
| 6 | XGBoost | 0.5024 | 0.4643 | 0.5474 | **0.8179** | 0.8248 |
| 7 | Balanced RF | 0.5000 | 0.5294 | 0.4737 | 0.8041 | 0.8469 |
| 8 | Random Forest | 0.5000 | 0.5294 | 0.4737 | 0.8041 | 0.8469 |
| - | Baseline | 0.0000 | — | 0.0000 | 0.5000 | 0.8384 |

**Najlepszy model (F1): Logistic Regression** — F1=0.5775, próg OOF=0.557  
Macierz pomyłek: TN=455, FP=38, FN=41, **TP=54** (z 95 rzeczywistych odejść)

### Architektura pipeline

| Model | SMOTE | Scaler | Balansowanie |
|-------|-------|--------|-------------|
| Logistic Regression | TAK | TAK | SMOTE |
| Balanced RF | TAK | NIE | SMOTE + BalancedRF |
| Random Forest | TAK | NIE | SMOTE |
| XGBoost | TAK | NIE | SMOTE + scale_pos_weight |
| SVM | TAK | TAK | SMOTE |
| MLP (sklearn) | TAK | TAK | SMOTE |
| **CatBoost** | NIE | NIE | auto_class_weights='Balanced' |
| **Stacking** | NIE | NIE | base: własne balansowanie |

### Stacking — meta-learner

Stacking łączy 3 modele bazowe (BRF 60 drzew, XGB 80 drzew, CatBoost 60 iter.) i trenuje meta-klasyfikator (Logistic Regression) na ich predykcjach probabilistycznych (3-fold cross-val).

```
BRF (60) ──┐
XGB (80) ──┼──► [meta-features 3-fold CV] ──► LR meta ──► P(Attrition=1)
CB  (60) ──┘
```

Próg stackingu dobrany na **zbiorze walidacyjnym** (val=177 próbek) — unika zagnieżdżonego CV.

### Próg decyzyjny

- Modele 1–8: próg z **OOF (Out-of-Fold)** — uczciwy estymator bez wycieku danych testowych
- Stacking: próg z **val set** — unika zagnieżdżonego CV (stacking wewnętrznie robi już 3-fold)

### Wnioski biznesowe

1. **Logistic Regression wygrywa** — F1=0.5775, najprostszy model, najlepsza generalizacja
2. **Stacking: najwyższy Recall=65%** — wykrywa 62 z 95 odejść, ale 81 fałszywych alarmów vs 38 przy LR
3. **SVM: najwyższa precyzja** (64.6%) i dokładność (87%) — optymalny gdy koszt fałszywego alarmu wysoki
4. **XGBoost: najlepsze ROC-AUC** (0.8179) — najlepsza separacja klas w spektrum progów
5. Cel F1 ≥ 0.80 nierealistyczny przy ~16% Attrition; realny pułap: **F1 ~0.56–0.65**

### Segmentacja — uzupełnienie modelu predykcyjnego

KMeans (k=4) na 21 cechach identyfikuje 4 segmenty pracowników:

| Klaster | N (%) | Attrition | Priorytety HR |
|---------|-------|-----------|---------------|
| Nomadzi | 447 (30%) | 19% | Ścieżka kariery, mentoring |
| **Starterzy** | 431 (29%) | **22%** | Podwyżki entry-level, buddy |
| Lojalni Stagnujący | 407 (28%) | 10% | Lateral moves, projekty |
| Seniorzy | 185 (13%) | 9% | Mentoring odwrócony, elastyczność |